# Lab 2 — Linear Maps and Their Matrices
**Linear Algebra · UATX**  
*Week 3 · Prerequisites: §1.5 (pages 11–14)*

Every linear map $T: V \to W$ between finite-dimensional spaces is completely determined by its action on a basis: if $\mathcal{B} = \{e_1,\ldots,e_n\}$ is a basis of $V$, the **matrix of $T$** is
$$M_T = [T(e_1) \mid T(e_2) \mid \cdots \mid T(e_n)].$$
Composition of linear maps corresponds to matrix multiplication: $M_{S \circ T} = M_S M_T$.

**Tasks**
1. Build differentiation $D$ and integration $J$ as explicit matrices on $P_n$; verify $DJ = I$.
2. Write `matrix_of_T(T, basis)` for arbitrary linear maps.
3. Compute matrices of reflection, rotation, and projection.
4. Verify $M_{S \circ T} = M_S M_T$ for two composed maps.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
np.random.seed(0)

## §1  Differentiation and Integration as Linear Maps

Work in $P_n$ with the monomial basis $\{1, x, x^2, \ldots, x^n\}$, identifying $p = \sum_{k=0}^n c_k x^k$ with the coefficient vector $c = [c_0,\ldots,c_n]^\top \in \mathbb{R}^{n+1}$.

**Differentiation** $D: P_n \to P_{n-1}$: $D(x^k) = k x^{k-1}$ for $k \geq 1$, $D(1) = 0$.  
**Integration** $J: P_{n-1} \to P_n$: defined by $J(p)(x) = \int_0^1 x \, p(xt) \, dt$.

In [ ]:
def diff_matrix(n):
    """
    Matrix of D: P_n -> P_{n-1} in the monomial basis.
    Returns a (n) x (n+1) matrix (maps R^{n+1} -> R^n).
    D(x^k) = k x^{k-1}; in coordinates: c -> [c[1], 2c[2], ..., n*c[n]]
    """
    D = np.zeros((n, n+1))
    for k in range(1, n+1):
        D[k-1, k] = k   # coefficient of x^{k-1} in D(x^k) is k
    return D


def int_matrix(n):
    """
    Matrix of J: P_{n-1} -> P_n in the monomial basis.
    J(x^k)(x) = int_0^1 x * (xt)^k dt = x^{k+1} * int_0^1 t^k dt = x^{k+1} / (k+1).
    So J maps e_k (in R^n) to e_{k+1} / (k+1) (in R^{n+1}).
    Returns a (n+1) x n matrix.
    """
    J = np.zeros((n+1, n))
    for k in range(n):
        J[k+1, k] = 1.0 / (k+1)
    return J


n = 4
D = diff_matrix(n)
J = int_matrix(n)

print(f'D  ({D.shape[0]}x{D.shape[1]}) = differentiation on P_{n}:')
print(D)
print(f'\nJ  ({J.shape[0]}x{J.shape[1]}) = integration on P_{n-1}:')
print(J)

In [ ]:
# Verify D(x^2) = 2x  (coefficient vector [0,0,1,0,0] -> [0,2,0,0])
e2 = np.zeros(n+1); e2[2] = 1.0
print('D(x^2) =', D @ e2, '  expected [0, 2, 0, 0]')

# Verify D(x^3) = 3x^2
e3 = np.zeros(n+1); e3[3] = 1.0
print('D(x^3) =', D @ e3, '  expected [0, 0, 3, 0]')

# Verify D ∘ J = I_{n x n}
DJ = D @ J
print('\nD @ J (should be identity I_{n}):')
print(np.round(DJ, 10))
print('D∘J = I:', np.allclose(DJ, np.eye(n)))

In [ ]:
# Visualise: apply J then D to a polynomial p
coeffs_p = np.array([1., -2., 0., 3., 0.])  # p(x) = 1 - 2x + 3x^3
print('p  coefficients:', coeffs_p)
print('Jp coefficients:', J @ coeffs_p[:n],'  (J maps P_{n-1} inputs; using first n coefficients)')

# Better demonstration: work entirely in P_{n-1}
coeffs_q = np.array([2., -1., 3., 4.])   # q in P_{n-1}
Jq = J @ coeffs_q
DJq = D @ Jq
print('\nq  =', coeffs_q)
print('Jq =', Jq)
print('DJq=', DJq, '  should equal q:', np.allclose(DJq, coeffs_q))

## §2  The Matrix of a Linear Map

For any linear map $T: \mathbb{R}^n \to \mathbb{R}^m$, the matrix is $M_T = [T(e_1) \mid \cdots \mid T(e_n)]$.  
This works for *any* ordered basis of the domain.

In [ ]:
def matrix_of_T(T, basis):
    """
    Build the matrix of a linear map T w.r.t. the given ordered basis.
    T      : callable, maps a vector to a vector
    basis  : list of n column vectors (each an np.ndarray)
    Returns: matrix M such that M @ [v]_basis = T(v)  (in standard coordinates)
    """
    return np.column_stack([T(b) for b in basis])


# Standard basis of R^2
e1, e2 = np.array([1.,0.]), np.array([0.,1.])
basis_2d = [e1, e2]

# --- Reflection across y = x ---
reflect_yx = lambda v: np.array([v[1], v[0]])
M_ref = matrix_of_T(reflect_yx, basis_2d)
print('Reflection across y=x:')
print(M_ref)

# --- Rotation by theta ---
def rotation(theta):
    c, s = np.cos(theta), np.sin(theta)
    return lambda v: np.array([c*v[0]-s*v[1], s*v[0]+c*v[1]])

M_rot = matrix_of_T(rotation(np.pi/4), basis_2d)
print('\nRotation by pi/4:')
print(np.round(M_rot, 6))

# --- Orthogonal projection onto direction n_vec ---
n_vec = np.array([1., 2.]) / np.sqrt(5)
project_onto = lambda v: np.dot(v, n_vec) * n_vec
M_proj = matrix_of_T(project_onto, basis_2d)
print('\nProjection onto (1,2)/sqrt(5):')
print(np.round(M_proj, 6))
print('P^2 = P (idempotent):', np.allclose(M_proj @ M_proj, M_proj))

In [ ]:
# Visualise the effect of each map on the unit circle
theta = np.linspace(0, 2*np.pi, 300)
circle = np.vstack([np.cos(theta), np.sin(theta)])

fig, axes = plt.subplots(1, 3, figsize=(14, 4))
for ax, M, title in zip(axes,
    [M_ref, M_rot, M_proj],
    ['Reflection across $y=x$', 'Rotation by $\\pi/4$', 'Projection onto $(1,2)/\\sqrt{5}$']):
    img = M @ circle
    ax.plot(circle[0], circle[1], 'b-', label='original', alpha=0.5)
    ax.plot(img[0], img[1], 'r-', label='image', linewidth=2)
    ax.set_aspect('equal'); ax.grid(True); ax.legend(); ax.set_title(title)
plt.tight_layout(); plt.show()

## §3  Composition = Matrix Multiplication

**Theorem.** If $T_1: \mathbb{R}^n \to \mathbb{R}^m$ and $T_2: \mathbb{R}^m \to \mathbb{R}^p$ are linear, then $M_{T_2 \circ T_1} = M_{T_2} \cdot M_{T_1}$.

In [ ]:
# T1 = rotation by pi/6, T2 = reflection across y=x
T1 = rotation(np.pi / 6)
T2 = reflect_yx
T1_then_T2 = lambda v: T2(T1(v))

M1 = matrix_of_T(T1, basis_2d)
M2 = matrix_of_T(T2, basis_2d)

# Method A: matrix product
M_composed_matrix = M2 @ M1

# Method B: matrix of the composed function
M_composed_direct = matrix_of_T(T1_then_T2, basis_2d)

print('M2 @ M1 ='); print(np.round(M_composed_matrix, 8))
print('\nmatrix_of_T(T2∘T1) ='); print(np.round(M_composed_direct, 8))
print('\nEqual:', np.allclose(M_composed_matrix, M_composed_direct))

In [ ]:
# Test associativity: (M3 M2) M1 = M3 (M2 M1)
T3 = rotation(np.pi / 3)
M3 = matrix_of_T(T3, basis_2d)
print('(M3 @ M2) @ M1 = M3 @ (M2 @ M1):', np.allclose((M3 @ M2) @ M1, M3 @ (M2 @ M1)))

# Test on differentiation + integration in P_n
print('\nAssociativity for D, J matrices:')
n2 = 3
D2, J2 = diff_matrix(n2), int_matrix(n2)
A = np.random.randn(n2+1, n2+1)
print('(D @ J) @ A = D @ (J @ A):', np.allclose((D2 @ J2) @ np.eye(n2), D2 @ (J2 @ np.eye(n2))))

## §4  Non-square Example: $D: P_3 \to P_2$

Visualize the matrix of $D$ on $P_3$ (4-dim) mapping to $P_2$ (3-dim).

In [ ]:
D3 = diff_matrix(3)
print('D: P_3 -> P_2  (shape', D3.shape, '):')
print(D3)

# The space L(P_3, P_2) of linear maps is itself a vector space of dimension 3*4=12
# Addition of maps: (S+T)(p) = S(p) + T(p)
D3_double = 2 * D3
print('\n2D (doubling the derivative map):')
print(D3_double)

# Apply to p(x) = 1 + x + x^2 + x^3
p_test = np.ones(4)
print('\np(x) = 1+x+x^2+x^3  coefficients:', p_test)
print('Dp  = p\'(x) coefficients:', D3 @ p_test,'  (should be [1,2,3])')